# Evaluation of semantic search and reranking 

In this notebook section it will be assessed the quality of the semantic search tool on a little portion of hand‑labeled test queries. For each query we compare these two systems: vector‑only retrieval using sentence embeddings and retrieval with the cross‑encoder reranker plus the difficulty metadata bonus. It computes standard information retrieval metrics (Precision@5 and Mean Reciprocal Rank, MRR) for both systems in order to see whether reranking (and metadata) actually improve the ranking quality for realistic user queries.

### Qualitative examples from the Streamlit demo

To complement the quantitative metrics, we also inspected the ranked lists returned by the Streamlit demo.

For the beginner-style query *"beginner rules about learning Python"*, the top results are:

1. **python help** – a prompt about learning Python from scratch  
2. **want to learn ml where do i even start** – a prompt about getting started in ML with only basic Python  
3. **Could you help an old lady understand something?** – a prompt where a 75‑year‑old learner asks about pytest  

All three are clearly oriented toward beginners and learning, which matches the intent of the query and is consistent with the difficulty‑aware reranking.

For the more advanced query *"improve to formal this email for my professor"*, the top results are:

1. **Rewrite a formal email in multiple tones**  
2. **Translate Business Email Formally**  
3. **email to my prof**  

These prompts all focus on formal or professional email rewriting, which aligns well with the user’s request and explains why the reranker achieves a higher MRR on this query in the quantitative evaluation.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parents[0]
sys.path.append(str(PROJECT_ROOT / "src"))

from search import search

In [5]:
test_queries = [
    {
        "query": "Explain research in simple words to my grandma",
        "relevant_ids": ["pk_11545", "pk_10388"],
    },
    {
        "query": "improve to formal this email for my professor",
        "relevant_ids": ["pk_07054", "pk_05274", "pk_08737", "pk_18262"],
    },
]

Vector-only search and vector + reranker + difficulty metadata 

In [ ]:
def run_search_for_query(query_text, k=5):

    res_vec = search(query_text, top_k=k, use_reranker=False)
    ids_vec = res_vec["ids"][0]

    res_rerank = search(query_text, top_k=k, use_reranker=True)
    ids_rerank = res_rerank["ids"][0]

    return ids_vec, ids_rerank

In [7]:
q = test_queries[0]
ids_vec, ids_rerank = run_search_for_query(q["query"], k=5)
ids_vec, ids_rerank

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9868.73it/s]


Loading reranker model: cross-encoder/ms-marco-MiniLM-L-6-v2


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 9389.23it/s]


(['pk_11545', 'pk_10290', 'pk_10388', 'pk_02947', 'pk_11772'],
 ['pk_11545', 'pk_10388', 'pk_02947', 'pk_10290', 'pk_11772'])

In [8]:
def precision_at_k(pred_ids, relevant_ids, k):
    pred_top_k = pred_ids[:k]
    hits = sum(1 for pid in pred_top_k if pid in relevant_ids)
    return hits / k

def reciprocal_rank(pred_ids, relevant_ids):
    for idx, pid in enumerate(pred_ids, start=1):
        if pid in relevant_ids:
            return 1.0 / idx
    return 0.0

Precision@5 (vector-only vs rerank): both 0.4 → 2 relevant out of 5 in each case.

MRR (vector-only vs rerank + difficulty): both 1.0 → in both rankings, the first result is relevant, so reciprocal rank is 1 / 1.

In [9]:
q = test_queries[0]
relevant = q["relevant_ids"]

p_vec = precision_at_k(ids_vec, relevant, k=5)
p_rerank = precision_at_k(ids_rerank, relevant, k=5)

rr_vec = reciprocal_rank(ids_vec, relevant)
rr_rerank = reciprocal_rank(ids_rerank, relevant)

p_vec, p_rerank, rr_vec, rr_rerank

(0.4, 0.4, 1.0, 1.0)

In [ ]:
def evaluate_all(test_queries, k=5):
    results = []

    for tq in test_queries:
        query = tq["query"]
        relevant = tq["relevant_ids"]

        ids_vec, ids_rerank = run_search_for_query(query, k=k)

        p_vec = precision_at_k(ids_vec, relevant, k)
        p_rerank = precision_at_k(ids_rerank, relevant, k)

        rr_vec = reciprocal_rank(ids_vec, relevant)
        rr_rerank = reciprocal_rank(ids_rerank, relevant)

        results.append({
            "query": query,
            "P_vec": p_vec,
            "P_rerank": p_rerank,
            "RR_vec": rr_vec,
            "RR_rerank": rr_rerank,
        })


    avg_p_vec = sum(r["P_vec"] for r in results) / len(results)
    avg_p_rerank = sum(r["P_rerank"] for r in results) / len(results)
    avg_rr_vec = sum(r["RR_vec"] for r in results) / len(results)
    avg_rr_rerank = sum(r["RR_rerank"] for r in results) / len(results)

    return results, {
        "avg_P_vec": avg_p_vec,
        "avg_P_rerank": avg_p_rerank,
        "avg_RR_vec": avg_rr_vec,
        "avg_RR_rerank": avg_rr_rerank,
    }

In [11]:
per_query, averages = evaluate_all(test_queries, k=5)
per_query, averages

([{'query': 'Explain research in simple words to my grandma',
   'P_vec': 0.4,
   'P_rerank': 0.4,
   'RR_vec': 1.0,
   'RR_rerank': 1.0},
  {'query': 'improve to formal this email for my professor',
   'P_vec': 0.8,
   'P_rerank': 0.8,
   'RR_vec': 0.5,
   'RR_rerank': 1.0}],
 {'avg_P_vec': 0.6000000000000001,
  'avg_P_rerank': 0.6000000000000001,
  'avg_RR_vec': 0.75,
  'avg_RR_rerank': 1.0})

In [12]:
from pprint import pprint

print("Per‑query metrics:")
for r in per_query:
    print("\nQuery:", r["query"])
    print(f"  Precision@5  - vector: {r['P_vec']:.2f}  | rerank: {r['P_rerank']:.2f}")
    print(f"  MRR          - vector: {r['RR_vec']:.2f} | rerank: {r['RR_rerank']:.2f}")

print("\nAverage metrics across queries:")
print(f"  Precision@5  - vector: {averages['avg_P_vec']:.2f}  | rerank: {averages['avg_P_rerank']:.2f}")
print(f"  MRR          - vector: {averages['avg_RR_vec']:.2f} | rerank: {averages['avg_RR_rerank']:.2f}")

Per‑query metrics:

Query: Explain research in simple words to my grandma
  Precision@5  - vector: 0.40  | rerank: 0.40
  MRR          - vector: 1.00 | rerank: 1.00

Query: improve to formal this email for my professor
  Precision@5  - vector: 0.80  | rerank: 0.80
  MRR          - vector: 0.50 | rerank: 1.00

Average metrics across queries:
  Precision@5  - vector: 0.60  | rerank: 0.60
  MRR          - vector: 0.75 | rerank: 1.00


## Results

Vector‑only retrieval and reranker + difficulty have the same Precision@5 (0.60 on average). This signifies that these retrieve a similar number of relevant prompts in the top 5 rank. The reranker  improves the Mean Reciprocal Rank (MRR increases from 0.75 to 1.00) by much as it tends to move the most relevant prompts on top of the rank with emphasis for the email‑improvement one. So while the embedding model retrieves well by combining also the cross‑encoder reranker and the difficulty‑aware bonus it produces rankings that are more useful for each user